# 02. River HalfSpaceTrees 실시간 탐지

목표: CPU, memory, error rate와 p95 latency를 한 건씩 점수화하는 online detector를 만듭니다.

설치: `python -m pip install -r requirements.txt`

중요: `score_one()`으로 먼저 평가하고 정상으로 판단한 관측만 `learn_one()`에 넣습니다.

In [ ]:
import math
import random
import river
from river import anomaly

print("river version:", river.__version__)
random.seed(7)

In [ ]:
def make_stream(size=400):
    rows = []
    for t in range(size):
        row = {
            "t": t,
            "features": {
                "cpu_pct": 38 + 9 * math.sin(t / 23) + random.gauss(0, 2.5),
                "memory_pct": 55 + 3 * math.sin(t / 70) + random.gauss(0, 1.0),
                "error_rate": max(0.0, 0.008 + random.gauss(0, 0.003)),
                "p95_latency_ms": 140 + 20 * math.sin(t / 17) + random.gauss(0, 7),
            },
            "is_anomaly": False,
        }
        if 200 <= t < 210:
            row["features"].update({
                "cpu_pct": 94 + random.gauss(0, 1),
                "memory_pct": 88 + random.gauss(0, 1),
                "error_rate": 0.35 + random.random() * 0.1,
                "p95_latency_ms": 1350 + random.gauss(0, 50),
            })
            row["is_anomaly"] = True
        if 330 <= t < 335:
            row["features"]["p95_latency_ms"] = 1750 + random.gauss(0, 40)
            row["features"]["error_rate"] = 0.18 + random.random() * 0.04
            row["is_anomaly"] = True
        rows.append(row)
    return rows

stream = make_stream()
print("observations:", len(stream), "anomalies:", sum(row["is_anomaly"] for row in stream))

## Model과 calibration

각 feature의 물리적 범위를 명시합니다. 정상으로 가정한 첫 160개 관측을 학습하고 score의 99 percentile을 threshold 후보로 사용합니다.

In [ ]:
limits = {
    "cpu_pct": (0.0, 100.0),
    "memory_pct": (0.0, 100.0),
    "error_rate": (0.0, 1.0),
    "p95_latency_ms": (0.0, 2000.0),
}

model = anomaly.HalfSpaceTrees(
    n_trees=15,
    height=6,
    window_size=50,
    limits=limits,
    seed=42,
)

def percentile(values, q):
    ordered = sorted(values)
    position = (len(ordered) - 1) * q
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

calibration_end = 160
calibration_scores = []
for row in stream[:calibration_end]:
    score = model.score_one(row["features"])
    model.learn_one(row["features"])
    if row["t"] >= 50:
        calibration_scores.append(score)

threshold = max(0.20, percentile(calibration_scores, 0.99))
print(f"calibration score range={min(calibration_scores):.3f}..{max(calibration_scores):.3f}")
print(f"threshold={threshold:.3f}")

## 미래 stream 평가

threshold 초과 관측은 모델에 학습시키지 않습니다. production에서는 즉시 폐기하지 말고 quarantine queue에 저장한 뒤 운영자 판정에 따라 지연 학습하는 방식이 안전합니다.

In [ ]:
scores = [0.0] * calibration_end
raw_flags = [False] * calibration_end

for row in stream[calibration_end:]:
    score = model.score_one(row["features"])
    is_alert = score > threshold
    scores.append(score)
    raw_flags.append(is_alert)
    if not is_alert:
        model.learn_one(row["features"])

def persistence_gate(flags, window=3, required=2):
    result = []
    for index in range(len(flags)):
        recent = flags[max(0, index - window + 1):index + 1]
        result.append(sum(recent) >= required)
    return result

confirmed = persistence_gate(raw_flags)
alert_rows = [(row["t"], round(scores[row["t"]], 3), row["is_anomaly"]) for row in stream if confirmed[row["t"]]]
print("confirmed alert points:", len(alert_rows))
print("first alerts:", alert_rows[:12])

In [ ]:
def metrics(labels, predictions):
    tp = sum(label and prediction for label, prediction in zip(labels, predictions))
    fp = sum((not label) and prediction for label, prediction in zip(labels, predictions))
    fn = sum(label and (not prediction) for label, prediction in zip(labels, predictions))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}

labels = [row["is_anomaly"] for row in stream]
print("raw:", metrics(labels, raw_flags))
print("persistent:", metrics(labels, confirmed))

실제 환경에서는 `limits`, tree 수, height, window와 threshold를 latency·memory·alert budget과 함께 기록하세요. anomaly가 한 window에 계속 몰리는 장애는 HalfSpaceTrees의 알려진 약점이므로 MAD, SLO burn rate와 병렬 비교하는 것이 좋습니다.